In [ ]:
import pandas as pd
import numpy as np

from sqlalchemy import create_engine

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error

from xgboost import XGBRegressor

In [18]:
connection_string = (
    "mssql+pyodbc://@localhost/mlb?"
    "driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"
)

engine = create_engine(connection_string)

query = """
select * from mlb.dbo.fact_hitter_pitcher_matchup_model_features
"""

df = pd.read_sql(query, engine)
print(df.shape)
df.head()

(24641, 499)


,gamePk,game_date,season,hitter_id,hitter_name,hitter_position,hitter_team_id,hitter_team_name,pitcher_id,pitcher_name,...,pitcher_weighted_csw_rate_last_10,pitcher_weighted_sc_strike_rate_last_10,pitcher_weighted_velocity_last_10,pitcher_weighted_spin_rate_last_10,pitcher_weighted_chase_rate_last_10,pitcher_weighted_putaway_rate_last_10,pitcher_prev_whiff_rate,pitcher_prev_csw_rate,pitcher_prev_chase_rate,pitcher_strikeOuts
0,778068,2025-05-04,2025,516782,Starling Marte,LF,118,Kansas City Royals,669467,Andre Pallante,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
1,823401,2026-04-06,2026,687462,Spencer Horwitz,1B,134,Pittsburgh Pirates,608566,Germán Márquez,...,NaN,NaN,NaN,NaN,NaN,NaN,0.030769,0.261538,0.269231,4
2,824698,2026-04-01,2026,518595,Travis d'Arnaud,C,108,Los Angeles Angels,571510,Matthew Boyd,...,NaN,NaN,NaN,NaN,NaN,NaN,0.285714,0.412698,0.297297,10
3,777137,2025-07-12,2025,543309,Kyle Higashioka,C,140,Texas Rangers,664285,Framber Valdez,...,0.280093,0.465605,88.337834,2406.148003,0.332585,0.210342,0.078431,0.235294,0.294118,10
4,776969,2025-07-28,2025,605137,Josh Bell,1B,142,Minnesota Twins,664285,Framber Valdez,...,0.272823,0.466111,88.458581,2395.127776,0.308990,0.209552,0.096774,0.225806,0.250000,12


In [19]:
# checking columns
print (df.columns.tolist())

['gamePk', 'game_date', 'season', 'hitter_id', 'hitter_name', 'hitter_position', 'hitter_team_id', 'hitter_team_name', 'pitcher_id', 'pitcher_name', 'pitcher_team_id', 'pitcher_team_name', 'pitcher_throws', 'hitter_stand', 'hitter_strikeOuts', 'pitches_seen_vs_pitcher', 'swings_vs_pitcher', 'whiffs_vs_pitcher', 'called_strikes_vs_pitcher', 'matchup_whiff_rate', 'matchup_called_strike_rate', 'matchup_csw_rate', 'hitter_days_since_last_game', 'hitter_avg_k_last_3', 'hitter_avg_pa_last_3', 'hitter_avg_ab_last_3', 'hitter_avg_hits_last_3', 'hitter_avg_hr_last_3', 'hitter_avg_bb_last_3', 'hitter_avg_pitches_last_3', 'hitter_avg_tb_last_3', 'hitter_avg_rbi_last_3', 'hitter_avg_lob_last_3', 'hitter_avg_obp_last_3', 'hitter_avg_slg_last_3', 'hitter_avg_ops_last_3', 'hitter_avg_babip_last_3', 'hitter_avg_batting_avg_last_3', 'hitter_avg_hbp_last_3', 'hitter_avg_sf_last_3', 'hitter_avg_sbunts_last_3', 'hitter_avg_stolen_bases_last_3', 'hitter_avg_caught_stealing_last_3', 'hitter_avg_k_rate_last_

In [20]:
# define features and targets
# X = features & y = target

target = "pitcher_strikeOuts"

drop_cols = [
    "gamePk",
    "game_date",
    "hitter_name",
    "pitcher_name",
    "hitter_position",
    "hitter_team_name",
    "pitcher_team_name"
]


leakage_cols = [
    "hitter_strikeOuts",
    "pitches_seen_vs_pitcher",
    "swings_vs_pitcher",
    "whiffs_vs_pitcher",
    "called_strikes_vs_pitcher",
    "matchup_whiff_rate",
    "matchup_called_strike_rate",
    "matchup_csw_rate"
]

X = df.drop(columns=drop_cols + leakage_cols + [target])
y = df[target]

print(X.shape)
print(y.shape)


(24641, 483)
(24641,)


In [ ]:
important_features = [
    # ===== Pitcher form (WEIGHTED) =====
    "pitcher_weighted_k_last_3",
    "pitcher_weighted_k_last_5",
    "pitcher_prev_k",

    # ===== Opportunity =====
    "pitcher_avg_bf_last_3",
    "pitcher_avg_ip_last_3",
    "pitcher_avg_pitches_last_3",
    "pitcher_avg_outs_last_3",
    "pitcher_gamesStarted",

    # ===== Skill =====
    "pitcher_weighted_whiff_rate_last_3",
    "pitcher_avg_velocity_last_3",
    "pitcher_avg_putaway_rate_last_3",

    # ===== Context =====
    "pitcher_throws",
    "hitter_stand"
]

X = X[important_features]
print(X.shape)

(24641, 13)


In [24]:

from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("MAE:", mae)
print("RMSE:", rmse)

MAE: 1.8016008138656616
RMSE: 2.2329995783708503


In [25]:
train_df = df[df["season"] == 2025]
test_df = df[df["season"] == 2026]

X_train = train_df[important_features]
y_train = train_df[target]

X_test = test_df[important_features]
y_test = test_df[target]

print(X_train.shape, X_test.shape)

(20905, 13) (3736, 13)


In [26]:
X_train = pd.get_dummies(X_train, columns=["pitcher_throws", "hitter_stand"], drop_first=True)
X_test  = pd.get_dummies(X_test,  columns=["pitcher_throws", "hitter_stand"], drop_first=True)

# Align columns (VERY IMPORTANT)
X_train, X_test = X_train.align(X_test, join="left", axis=1, fill_value=0)

In [27]:
model = XGBRegressor(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42
)

model.fit(X_train, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes

In [28]:
y_pred = model.predict(X_test)

print(y_pred[:10])

[5.492429  4.538693  4.9547963 4.9324064 1.3448085 1.7816412 4.6362247
 5.19545   5.6189237 6.3810344]


In [30]:

from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("MAE:", mae)
print("RMSE:", rmse)

MAE: 1.6739062070846558
RMSE: 2.079342592959728
